# NLP Exercises (Part 2)

We have 2 exercises in this section. The exercises are:

4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [1]:
# imports
import numpy as np
import re
import random
import nltk

In [2]:
# data

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]

nltk.download('gutenberg')
nltk.download('genesis')
nltk.download('inaugural')
nltk.download('nps_chat')
nltk.download('webtext')
nltk.download('treebank')
from nltk.book import *

wall_street = text3.tokens
tokens = wall_street

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package genesis to /root/nltk_data...
[nltk_data]   Package genesis is already up-to-date!
[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!
[nltk_data] Downloading package nps_chat to /root/nltk_data...
[nltk_data]   Package nps_chat is already up-to-date!
[nltk_data] Downloading package webtext to /root/nltk_data...
[nltk_data]   Package webtext is already up-to-date!
[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


In [3]:
# Bag of Words class implementation
class BagOfWords:
    def __init__(self):
        self.__feature_names = []

    # helper method
    # copied from Jan_Kwinta_exercises_7.ipynb
    def __tokenize(self, text: str) -> list:
        return re.findall(r'\w+', text)

    # build matrix
    def fit_transform(self, corpus: list) -> np.array:
        tokenized_corpus = [self.__tokenize(doc) for doc in corpus]

        unique_words = set()
        for doc_tokens in tokenized_corpus:
            unique_words.update(doc_tokens)

        self.__feature_names = sorted(list(unique_words))

        bow_matrix = np.zeros((len(corpus), len(self.__feature_names)), dtype=int)

        word_to_index = {word: i for i, word in enumerate(self.__feature_names)}

        for i, doc_tokens in enumerate(tokenized_corpus):
            for word in doc_tokens:
                if word in word_to_index:
                    col_index = word_to_index[word]
                    bow_matrix[i, col_index] += 1

        return bow_matrix

    # get column labels
    def get_feature_names(self) -> list:
        return self.__feature_names

In [4]:
# print matrix and labels
vectorizer = BagOfWords()
X = vectorizer.fit_transform(corpus)

print(X)
print(vectorizer.get_feature_names())
print(f'There are {len(vectorizer.get_feature_names())} labels')

[[0 1 1 0 0 1 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0 1 0 0 0 1 1]
 [0 0 0 1 1 0 1 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 2 1 0 0 0 1 0 0]]
['As', 'Bag', 'Of', 'Really', 'This', 'Words', 'a', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'see', 'sparse', 'the', 'third', 'throughout', 'us', 'words', 'you']
There are 33 labels


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [5]:
def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match, tokens))
    return clean

tokens = cleanup()

# 5-gram model
def build_ngrams():
    ngrams = []
    for i in range(len(tokens)-N+1):
        ngrams.append(tokens[i:i+N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq  = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1

    return counts

def next_word(text, N, counts):
    token_seq = SEP.join(text.split()[-(N-1):])

    if token_seq not in counts:
        token_seq = random.choice(list(counts.keys()))

    choices = counts[token_seq].items()

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight
        if upto > r: return choice
    assert False # should not reach here

In [6]:
def clean_generated(text: str) -> str:
    text = re.sub(r'\s+([.!?])', r'\1', text)
    text = re.sub(r'(^|[.!?]\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    return text

N = 5
SEP = " "
sentence_count = 5

# model init
ngrams = build_ngrams()
counts = ngram_freqs(ngrams)

# random start sequence
start_seq = None

if start_seq is None:
    start_seq = random.choice(list(counts.keys()))

generated = start_seq.lower()

# generate
sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith(('.','!', '?')) else 0

final_text = clean_generated(generated)
print(final_text)

They were afraid. And Jacob their father said unto them Go again buy us a little food. And we said unto him We have dreamed a dream and there is no interpreter of it. And he spake unto Ephron in the audience of the children of Heth and Ephron the Hittite answered Abraham in the audience of the children of Ammon unto this day. And Abraham took sheep and oxen and menservants and womenservants and gave them unto Abraham and restored him Sarah his wife. And Abimelech said unto Abraham What mean these seven ewe lambs which thou hast set by themselves?
